# CNN Feature Extraction — BRACS
**AFFC-Net Base Paper Implementation**

Extracts ResNet-50 spatial feature maps `[N, 196, 1024]` from BRACS histopathology images.

**Dataset:** BRACS (Breast Carcinoma Subtyping) — 7 classes across train / val / test splits.
Classes: `0_N, 1_PB, 2_UDH, 3_FEA, 4_ADH, 5_DCIS, 6_IC`

**Method:** Identical to LC25000 pipeline — ResNet-50 truncated after `layer3` → `[B, 1024, 14, 14]` → reshaped to `[B, 196, 1024]`.

**Corrections vs original:**
- Direct `Resize(224,224)` instead of `Resize(256)→RandomCrop(224)` (paper §5.2.1 specifies 224×224 input)
- No augmentation at extraction time — features saved deterministically; augmentation belongs in training loop
- Labels saved separately as `bracs_labels_ver2.pt` aligned with the concatenated dataset order (train → val → test)

In [1]:
import os, time
import torch
import torchvision.models as models
from torchvision import datasets, transforms
from torch.utils.data import DataLoader, ConcatDataset
from PIL import Image

# ── GPU selection ──────────────────────────────────────────────────────────────
# Change to your GPU index or MIG partition string
DEVICE_ID = "cuda:1"
device = torch.device(DEVICE_ID if torch.cuda.is_available() else "cpu")
print("Using device:", device)
if device.type == "cuda":
    idx = int(DEVICE_ID.split(":")[-1]) if ":" in DEVICE_ID else 0
    print("GPU:", torch.cuda.get_device_name(idx))

Image.MAX_IMAGE_PIXELS = None  # allow large pathology images

Using device: cuda:1
GPU: NVIDIA H200


In [2]:
from PIL import ImageFile
ImageFile.LOAD_TRUNCATED_IMAGES = True

## Transform
Paper §5.2.1: *'resizing all input images to 224×224×3'*  
No augmentation here — features saved once, deterministically.

In [3]:
# Paper §5.2.1: resize directly to 224×224 (no crop at extraction time)
# Augmentation (random crop / flip) is applied during model training, not here.
transform = transforms.Compose([
    transforms.Resize((224, 224)),          # paper-exact: 224×224×3
    transforms.ToTensor(),
    transforms.Normalize(
        mean=[0.485, 0.456, 0.406],         # ImageNet stats (ResNet-50 pretrained)
        std=[0.229, 0.224, 0.225]
    )
])
print("Transform defined (224×224, no augmentation — deterministic extraction)")

Transform defined (224×224, no augmentation — deterministic extraction)


## Dataset
BRACS — 3892 images across 7 classes (0_N, 1_PB, 2_UDH, 3_FEA, 4_ADH, 5_DCIS, 6_IC).
Splits: Train (3010) / Val (312) / Test (570).

All three splits are concatenated **in order (train → val → test)** to extract features in one pass.
The split boundary indices are recorded so the GNN notebook can reconstruct the splits.

In [4]:
BASE_DIR = "BRACS_Dataset/BRACS_RoI/latest_version"   # ← change to your actual path if different

train_dataset = datasets.ImageFolder(
    root=os.path.join(BASE_DIR, "train"),
    transform=transform
)
val_dataset = datasets.ImageFolder(
    root=os.path.join(BASE_DIR, "val"),
    transform=transform
)
test_dataset = datasets.ImageFolder(
    root=os.path.join(BASE_DIR, "test"),
    transform=transform
)

dataset = ConcatDataset([train_dataset, val_dataset, test_dataset])

print("Train samples :", len(train_dataset))
print("Val samples   :", len(val_dataset))
print("Test samples  :", len(test_dataset))
print("Total samples :", len(dataset))
print("Classes       :", train_dataset.classes)
print("Class to idx  :", train_dataset.class_to_idx)

# Record split boundaries for downstream use
train_end = len(train_dataset)
val_end   = train_end + len(val_dataset)
test_end  = val_end   + len(test_dataset)
print(f"\nSplit indices → train: [0:{train_end}]  val: [{train_end}:{val_end}]  test: [{val_end}:{test_end}]")

Train samples : 3286
Val samples   : 312
Test samples  : 570
Total samples : 4168
Classes       : ['0_N', '1_PB', '2_UDH', '3_FEA', '4_ADH', '5_DCIS', '6_IC']
Class to idx  : {'0_N': 0, '1_PB': 1, '2_UDH': 2, '3_FEA': 3, '4_ADH': 4, '5_DCIS': 5, '6_IC': 6}

Split indices → train: [0:3286]  val: [3286:3598]  test: [3598:4168]


## ResNet-50 backbone — stop at layer3
Paper §4.1.1: *'after three stages, generated 14×14×1024 feature maps'*

In [5]:
loader = DataLoader(
    dataset,
    batch_size=128,        # large batch — GPU-bound extraction
    shuffle=False,         # keep original order for index alignment with GNN
    num_workers=8,
    pin_memory=True
)

# ResNet-50 truncated after layer3 → output [B, 1024, 14, 14]
_resnet = models.resnet50(weights="DEFAULT")
cnn_backbone = torch.nn.Sequential(
    _resnet.conv1,
    _resnet.bn1,
    _resnet.relu,
    _resnet.maxpool,
    _resnet.layer1,   # stage 1
    _resnet.layer2,   # stage 2
    _resnet.layer3    # stage 3 → [B, 1024, 14, 14]
    # layer4 NOT included — paper stops here
).to(device).eval()

print("CNN backbone ready  (ResNet-50, stops at layer3)")
print(f"Expected output shape per batch: [B, 1024, 14, 14]")

CNN backbone ready  (ResNet-50, stops at layer3)
Expected output shape per batch: [B, 1024, 14, 14]


## Feature extraction loop
Reshape `[B, 1024, 14, 14]` → `[B, 196, 1024]` (paper Eq.4, §5.2.2 shape for AFF input)

In [6]:
cnn_features_list = []
skipped = []

t0 = time.time()
global_idx = 0
with torch.no_grad():
    for batch_idx, (images, _) in enumerate(loader):
        try:
            images = images.to(device, non_blocking=True)
            feats  = cnn_backbone(images)          # [B, 1024, 14, 14]
            feats  = feats.flatten(2).permute(0, 2, 1).contiguous()
            cnn_features_list.append(feats.cpu())
            if batch_idx % 10 == 0:
                print(f"  Batch {batch_idx:>4}/{len(loader)}  shape={feats.shape}")
        except Exception as e:
            print(f"  ⚠ Skipped batch {batch_idx}: {e}")
            skipped.append(batch_idx)
        global_idx += 1

torch.cuda.synchronize()
elapsed = time.time() - t0
print(f"\nDone in {elapsed:.1f}s  |  skipped batches: {skipped}")

  Batch    0/33  shape=torch.Size([128, 196, 1024])
  Batch   10/33  shape=torch.Size([128, 196, 1024])
  Batch   20/33  shape=torch.Size([128, 196, 1024])
  Batch   30/33  shape=torch.Size([128, 196, 1024])

Done in 990.3s  |  skipped batches: []


In [10]:
cnn_features = torch.cat(cnn_features_list, dim=0)
print(f"Final CNN feature tensor shape : {cnn_features.shape}")
# Expected: [3892, 196, 1024]

assert cnn_features.shape[1] == 196,  "Expected 196 spatial nodes (14×14)"
assert cnn_features.shape[2] == 1024, "Expected 1024 feature channels"
print("Shape assertion passed.")

Final CNN feature tensor shape : torch.Size([4168, 196, 1024])
Shape assertion passed.


## Save features and labels
Labels are collected from the three ImageFolder datasets (train → val → test) in the same concatenated order.
Split boundary indices are also saved so downstream notebooks can reconstruct train/val/test splits.

In [11]:
# Collect labels in the same train → val → test order as ConcatDataset
labels = torch.tensor(
    train_dataset.targets + val_dataset.targets + test_dataset.targets,
    dtype=torch.long
)

assert len(labels) == len(cnn_features), \
    f"Label count {len(labels)} != feature count {len(cnn_features)}"

print(f"Labels shape : {labels.shape}")
print(f"Unique classes in labels: {labels.unique().tolist()}")
print(f"Class counts : { {c: (labels == c).sum().item() for c in labels.unique().tolist()} }")

Labels shape : torch.Size([4168])
Unique classes in labels: [0, 1, 2, 3, 4, 5, 6]
Class counts : {0: 484, 1: 836, 2: 322, 3: 756, 4: 507, 5: 630, 6: 633}


In [12]:
# Save features
torch.save(cnn_features, "bracs_cnn_features_ver2.pt")
print(f"Saved → bracs_cnn_features_ver2.pt  {cnn_features.shape}")

# Save labels
# At the end of CNN notebook, save with distinct name
torch.save(labels, "bracs_cnn_labels_ver2.pt")
print(f"Saved → bracs_cnn_labels_ver2.pt        {labels.shape}")

# Save split indices so GNN notebook can reconstruct train/val/test
split_info = {
    "train_end" : train_end,
    "val_end"   : val_end,
    "test_end"  : test_end,
    "classes"   : train_dataset.classes,
    "class_to_idx": train_dataset.class_to_idx
}
torch.save(split_info, "bracs_split_info_ver2.pt")
print(f"Saved → bracs_split_info_ver2.pt    {split_info}")

print("\nDone. Use bracs_labels_ver2.pt (NOT ImageFolder labels directly) in GNN notebook.")

Saved → bracs_cnn_features_ver2.pt  torch.Size([4168, 196, 1024])
Saved → bracs_cnn_labels_ver2.pt        torch.Size([4168])
Saved → bracs_split_info_ver2.pt    {'train_end': 3286, 'val_end': 3598, 'test_end': 4168, 'classes': ['0_N', '1_PB', '2_UDH', '3_FEA', '4_ADH', '5_DCIS', '6_IC'], 'class_to_idx': {'0_N': 0, '1_PB': 1, '2_UDH': 2, '3_FEA': 3, '4_ADH': 4, '5_DCIS': 5, '6_IC': 6}}

Done. Use bracs_labels_ver2.pt (NOT ImageFolder labels directly) in GNN notebook.
